**Release-candidate copy.** Sanitized (private paths / account handle redacted; see REDACTION_MANIFEST.md). A guard cell was prepended so nothing runs, downloads, or calls a paid API by accident. Executed outputs (if any) are the original run outputs.

In [ ]:
# RELEASE GUARD (added to the release copy; not part of the original run)
import os
if os.environ.get("WSP_ALLOW_NOTEBOOK_RUN") != "1":
    raise SystemExit("Saved results are in runs/local_pilot_phi4_mistral/. Re-running downloads ~90 GiB of model weights and needs 2 GPUs. Set WSP_ALLOW_NOTEBOOK_RUN=1 to proceed. No paid API is called.")


# WSP local-only pilot: Phi-4 14B + Mistral Small 24B
No provider API, secrets, baseline training or CIFAR. Two independent model families; 4-bit NF4 results, not full-precision claims. Exactly 3 sampled responses per model per task (modmul/permadd, no hint only), 12 planned responses. This is a feasibility pilot, NOT a publication-sized comparison. Fixed existing permadd mapping tests adaptation, not recall immunity. Save all attempts. No repairs or silent substitution.

Kaggle: Internet ON, GPU T4 x2; verify CURRENT quota before launch. Each card is separate 16GB memory, not a unified 32GB. Layers are explicitly sharded; CPU/disk model offload is rejected. Temporary scratch disk must fit original BF16 downloads even though inference is NF4. Models load sequentially and their private disposable download cache is removed between models. If disk/VRAM/compatibility fails, log SKIPPED/ERROR, never a model accuracy zero. Dependency install and real GPU generation remain unvalidated until Kaggle preflight. No secrets need attaching.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'accelerate==1.6.0', 'bitsandbytes==0.45.5', 'mistral-common==1.5.4', 'sentencepiece', 'protobuf', 'jinja2', 'pandas', 'matplotlib'])

In [ ]:
import os, json, time, math, random, platform, re, traceback, gc, tempfile, subprocess, sys, hashlib
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib; import matplotlib.pyplot as plt
import shutil, importlib.metadata
ON_KAGGLE = os.path.isdir('/kaggle/working')
OUT_DIR = os.environ.get('WSP_LOCAL_OUT', '/kaggle/working/local_strong_pilot' if ON_KAGGLE else '<STUDY_ROOT>/local_pilot/smoke_outputs')
os.makedirs(OUT_DIR, exist_ok=True)
GEN_DIR = OUT_DIR+'/generated_code'; os.makedirs(GEN_DIR, exist_ok=True)
DEVICE='cpu' # evaluation uses FP32 CPU; no contention with the synthesizer's GPU allocation
RUN_TASKS={'modmul':True,'permadd':True}
RUN_ORACLE_CHECK=True
SUCCESS_THRESH=.99
SANDBOX_TIMEOUT={'algo':120}
SPLIT=(.3,.1)
T_START=time.time()
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
MODELS=[
 {'id':'microsoft/phi-4','revision':'2db69c1c3e91a05d2c64a3185acfbaf36f744e25','disk_gib':35},
 {'id':'mistralai/Mistral-Small-24B-Instruct-2501','revision':'9527884be6e5616bdd54de542f9ae13384489724','disk_gib':54},
]
MAX_NEW_TOKENS=8192
MAX_GENERATION_S=900
MAX_TOTAL_S=6*3600
SAMPLING={'do_sample':True,'temperature':.6,'top_p':.95}
# Identical sampling is a deliberate pilot control, not Mistral's recommended .15.
PILOT_CASES=[(task,'none',rep) for rep in range(3) for task in ['modmul','permadd']]
# First one response per task is the operational smoke stage; remaining reps are still only a pilot.
print('LOCAL ONLY; no provider APIs. Planned responses:',len(MODELS)*len(PILOT_CASES),flush=True)


In [ ]:
P = 113
H_ALGO = 256
PAR_N, PAR_K, H_PAR = 16, 6, 64
PERM = np.random.default_rng(4321).permutation(113)   # secret relabelling for the recall-resistant task
PAR_SUBSET = sorted(np.random.default_rng(1234).choice(PAR_N, PAR_K, replace=False).tolist())

def primitive_root(p):
    fac = [q for q in range(2, p) if (p - 1) % q == 0 and all(q % r for r in range(2, int(q**.5) + 1))]
    return next(g for g in range(2, p) if all(pow(g, (p - 1) // q, p) != 1 for q in fac))
G_ROOT = primitive_root(P)

class MLP(nn.Module):
    """Shared architecture for A/B/C. Params: W1 (H,in), [b1 (H)], W2 (out,H), [b2 (out)]."""
    def __init__(self, d_in, H, d_out, act="quad", bias=False):
        super().__init__()
        self.act_name, self.bias = act, bias
        self.W1 = nn.Parameter(torch.randn(H, d_in) / math.sqrt(d_in))
        self.W2 = nn.Parameter(torch.randn(d_out, H) / math.sqrt(H))
        if bias: self.b1 = nn.Parameter(torch.zeros(H)); self.b2 = nn.Parameter(torch.zeros(d_out))
    def forward(self, x):
        h = x @ self.W1.T + (self.b1 if self.bias else 0)
        h = h * h if self.act_name == "quad" else F.relu(h)
        return h @ self.W2.T + (self.b2 if self.bias else 0)

def onehot2(a, b, p=P): return torch.cat([F.one_hot(a, p), F.one_hot(b, p)], -1).float()

def make_task(name):
    if name == "permadd":
        a, b = np.meshgrid(np.arange(P), np.arange(P), indexing="ij"); a, b = a.ravel(), b.ravel()
        inv = np.argsort(PERM); y = inv[(PERM[a] + PERM[b]) % P]
        X = onehot2(torch.tensor(a), torch.tensor(b)); Y = torch.tensor(y)
        shapes = {"W1": (H_ALGO, 2 * P), "W2": (P, H_ALGO)}
        return dict(name=name, X=X, Y=Y, shapes=shapes, n_cls=P, make_model=lambda: MLP(2 * P, H_ALGO, P, "quad", False))
    if name in ("modadd", "modmul"):
        lo = 0 if name == "modadd" else 1
        a, b = np.meshgrid(np.arange(lo, P), np.arange(lo, P), indexing="ij"); a, b = a.ravel(), b.ravel()
        y = (a + b) % P if name == "modadd" else (a * b) % P
        X = onehot2(torch.tensor(a), torch.tensor(b)); Y = torch.tensor(y)
        shapes = {"W1": (H_ALGO, 2 * P), "W2": (P, H_ALGO)}
        return dict(name=name, X=X, Y=Y, shapes=shapes, n_cls=P,
                    make_model=lambda: MLP(2 * P, H_ALGO, P, "quad", False))
    if name == "parity":
        idx = np.arange(2 ** PAR_N); bits = ((idx[:, None] >> np.arange(PAR_N)) & 1).astype(np.float32)
        y = bits[:, PAR_SUBSET].sum(1).astype(int) % 2
        shapes = {"W1": (H_PAR, PAR_N), "b1": (H_PAR,), "W2": (2, H_PAR), "b2": (2,)}
        return dict(name=name, X=torch.tensor(bits), Y=torch.tensor(y), shapes=shapes, n_cls=2,
                    make_model=lambda: MLP(PAR_N, H_PAR, 2, "relu", True))
    raise ValueError(name)

def splits(n, seed):
    perm = np.random.default_rng(seed).permutation(n); a, b = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
    return torch.tensor(perm[:a]), torch.tensor(perm[a:b]), torch.tensor(perm[b:])

# ---- Oracle constructors, kept as SOURCE strings so the mock synthesizer can return them as "LLM code" ----
ORACLE_SRC = {}
ORACLE_SRC["modadd"] = r"""
def build_weights():
    p, H, M = 113, 256, 3
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); x = np.arange(p); h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/p
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, :p] = np.cos(w*x+phi); W1[h, p:] = np.cos(w*x+psi); W2[:, h] = np.cos(w*x+phi+psi); h += 1
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
"""
ORACLE_SRC["modmul"] = r"""
def build_weights():
    p, H, M, g = 113, 256, 3, GROOT
    n = p - 1; log = np.zeros(p, dtype=int); v = 1
    for e in range(n): log[v] = e; v = (v*g) % p
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); L = log[1:]; h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/n
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, 1:p] = np.cos(w*L+phi); W1[h, p+1:] = np.cos(w*L+psi); W2[1:, h] = np.cos(w*L+phi+psi); h += 1
    W2[0, :] = 0.0
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
""".replace("GROOT", str(G_ROOT))
ORACLE_SRC["permadd"] = r"""
def build_weights():
    p, H, M = 113, 256, 3
    V = np.array(PERMLIST)
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/p
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, :p] = np.cos(w*V+phi); W1[h, p:] = np.cos(w*V+psi); W2[:, h] = np.cos(w*V+phi+psi); h += 1
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
""".replace("PERMLIST", str(PERM.tolist()))
ORACLE_SRC["parity"] = r"""
def build_weights():
    n, H, S = 16, 64, SUBSET
    k = len(S); W1 = np.zeros((H, n)); b1 = np.zeros(H); W2 = np.zeros((2, H)); b2 = np.array([1.0, -1.0])
    for j in range(k+1):
        W1[j, S] = 1.0; b1[j] = -j
        c = 2.0 if j == 0 else 4.0*(-1)**j
        W2[1, j] = c; W2[0, j] = -c
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
""".replace("SUBSET", str(PAR_SUBSET))

SYSTEM_PROMPT = ("You are an expert in mechanistic interpretability who writes neural network weights directly, "
                 "without any training, data fitting or gradients.")
FORMAT_RULES = """Return ONLY one python code block defining `def build_weights():` (no arguments) that uses only `np` (numpy, already
imported) and `math`; no other imports, no file or network access, no randomness unless seeded. It must return a dict mapping
each parameter name to a numpy array of exactly the shape listed. Do not print large arrays; compute them in code."""

def task_prompt(task, level):
    """Prompts for modadd/modmul/parity are byte-identical to v2 so v2 replies are valid cache entries."""
    if task == "modadd":
        spec = f"""Construct weights for a network computing (a+b) mod p, p={P}, for ALL p^2 pairs a,b in 0..{P-1}.
Architecture (fixed): x = concat(onehot(a), onehot(b)) (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h (elementwise square); logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias. argmax(logits) must equal (a+b) mod p."""
        hint = {"full": """Hint: Fourier features. With w_k = 2*pi*k/p, embed a and b as cos(w_k a + phase), cos(w_k b + phase'); squaring
yields cross terms cos(w_k(a+b)+phase+phase') via cos x cos y = [cos(x+y)+cos(x-y)]/2. Read out with W2[c,h] = cos(w_k c + phase+phase')
so logit_c ~ sum_k cos(w_k(a+b-c)). Use a grid of phases so unwanted terms (cos 2wa, cos 2wb, cos w(a-b), constants) cancel.""",
                "partial": "Hint: a solution exists using periodic (trigonometric) features and trig product identities.",
                "none": ""}[level]
    elif task == "modmul":
        spec = f"""Construct weights for a network computing (a*b) mod p, p={P}, for ALL pairs a,b in 1..{P-1} (0 never appears as input).
Architecture (fixed): x = concat(onehot(a), onehot(b)) with one-hot over 0..{P-1} (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h; logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias, logits indexed by c in 0..{P-1}. argmax(logits) must equal (a*b) mod p."""
        hint = {"full": f"""Hint: the multiplicative group mod p is cyclic of order {P-1} with generator g={G_ROOT}. Let log(a) be the discrete log base g.
Then a*b mod p corresponds to log(a)+log(b) mod {P-1}. Apply the Fourier construction for modular addition on Z_{P-1}: embed
cos(2*pi*k*log(a)/{P-1} + phase), square, read out with cos(2*pi*k*log(c)/{P-1} + phase+phase'), phase grid so unwanted terms cancel; row c=0 can be zero.""",
                "partial": "Hint: use the structure of the multiplicative group mod p (it is cyclic) and periodic features.",
                "none": ""}[level]
    elif task == "permadd":
        spec = f"""Symbols 0..{P-1} encode the elements of Z_{P} through a SECRET relabelling: symbol s stands for the integer VAL[s], where
VAL = {PERM.tolist()}.
Construct weights for a network that, given symbols a and b, outputs the symbol c with VAL[c] = (VAL[a] + VAL[b]) mod {P}, for ALL {P}^2 pairs.
Architecture (fixed): x = concat(onehot(a), onehot(b)) over symbols (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h (elementwise square); logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias, indexed by output symbol c."""
        hint = {"full": """Hint: Fourier features on the relabelled values. With w_k = 2*pi*k/p, embed symbol a as cos(w_k VAL[a] + phase), likewise b;
squaring yields cos(w_k(VAL[a]+VAL[b]) + phase+phase'). Read out with W2[c,h] = cos(w_k VAL[c] + phase+phase'); use a phase grid so unwanted terms cancel.""",
                "partial": "Hint: the relabelled operation is a cyclic group; periodic features of the underlying values work.",
                "none": ""}[level]
    elif task == "parity":
        spec = f"""Construct weights for a network computing the parity (XOR) of the input bits at positions S={PAR_SUBSET} (0-indexed) of a
{PAR_N}-bit input x in {{0,1}}^{PAR_N}, for ALL 2^{PAR_N} inputs; the other bits are distractors.
Architecture (fixed): h = relu(W1 @ x + b1), W1 ({H_PAR}, {PAR_N}), b1 ({H_PAR},); logits = W2 @ h + b2, W2 (2, {H_PAR}), b2 (2,).
Class 1 = odd parity, class 0 = even. argmax(logits) must be correct for every input."""
        hint = {"full": f"""Hint: let s = sum of the bits in S (an integer 0..{PAR_K}). Use units relu(s - j) for j=0..{PAR_K} and choose output weights
so the piecewise-linear function of s equals +1 at odd s and -1 at even s (a zig-zag); logits = [-f, f] plus bias.""",
                "partial": "Hint: the answer only depends on the count of active bits in S; a piecewise-linear function of that count suffices.",
                "none": ""}[level]
    shapes = TASKS[task]["shapes"]
    return f"{spec}\n{hint}\nParameter names and shapes: {json.dumps({k: list(v) for k, v in shapes.items()})}\n{FORMAT_RULES}"

TASKS = {t: make_task(t) for t in ["modadd", "modmul", "parity", "permadd"] if RUN_TASKS.get(t)}
for t, T in TASKS.items(): print(t, "N =", len(T["Y"]), "shapes", T["shapes"])
print("primitive root mod", P, "=", G_ROOT, "| parity subset", PAR_SUBSET)


In [ ]:
# ---------------- Sandboxed execution, robust code extraction, evaluation, metrics ----------------
# Fault-isolation harness (separate process + timeout + whitelist imports + audit hook). It is NOT a hardened security sandbox;
# all generated code is logged for review.
RUNNER = r'''
import sys, json, math, builtins as _b
import numpy as np
src_path, out_path, fn_name, in_path, want_torch = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5] == "1"
src = open(src_path).read()
INPUTS = dict(np.load(in_path))
import itertools, functools, collections, operator, cmath, fractions, random, typing, string, statistics, numbers, dataclasses, copy, heapq, bisect
import numpy.linalg, numpy.fft, numpy.random, numpy.polynomial, numpy.lib.stride_tricks
if want_torch:
    import torch, torch.nn, torch.nn.functional
ALLOWED_MODULES = {"numpy", "math", "itertools", "functools", "collections", "operator", "cmath", "fractions", "random", "typing",
                   "string", "statistics", "numbers", "dataclasses", "copy", "heapq", "bisect", "torch", "scipy"}
_real_import = _b.__import__
def _safe_import(name, globals=None, locals=None, fromlist=(), level=0):
    root = name.split(".")[0]
    if level != 0 or root not in ALLOWED_MODULES:
        raise ImportError(f"import of {name!r} is blocked in the sandbox")
    if root == "torch" and "torch" not in sys.modules:
        raise ImportError("torch not preloaded (pass want_torch)")
    return _real_import(name, globals, locals, fromlist, level)
SAFE_NAMES = ["abs","all","any","ascii","bin","bool","bytearray","bytes","callable","chr","complex","dict","divmod","enumerate",
  "filter","float","format","frozenset","hasattr","hash","hex","id","int","isinstance","issubclass","iter","len","list","map","max",
  "min","next","object","oct","ord","pow","print","range","repr","reversed","round","set","slice","sorted","str","sum","super",
  "tuple","type","zip","getattr","setattr","property","staticmethod","classmethod","NotImplemented","Ellipsis","None","True","False",
  "Exception","ValueError","TypeError","KeyError","IndexError","AssertionError","ArithmeticError","ZeroDivisionError",
  "RuntimeError","StopIteration","NotImplementedError","AttributeError","OverflowError","LookupError","ImportError","__build_class__"]
safe = {k: getattr(_b, k) for k in SAFE_NAMES if hasattr(_b, k)}
safe["__import__"] = _safe_import
def _blocked(name):
    def f(*a, **k): raise PermissionError(f"sandbox: builtin {name}() is blocked")
    return f
for _n in ["open", "eval", "exec", "compile", "input", "breakpoint", "globals", "locals", "vars", "memoryview", "exit", "quit", "help"]:
    safe[_n] = _blocked(_n)
_ALLOW = [False]
BLOCK_PREFIX = ("subprocess.", "os.system", "os.exec", "os.spawn", "os.fork", "os.posix_spawn", "os.kill", "os.remove", "os.rmdir",
                "os.rename", "os.unlink", "os.truncate", "os.chmod", "os.chown", "shutil.", "socket.", "urllib.", "http.", "ctypes.",
                "pty.", "fcntl.", "os.putenv", "os.chdir", "code.__new__")
def _hook(ev, args):
    if _ALLOW[0]: return
    if ev == "open":
        path, mode = str(args[0]), str(args[1] or "r")
        if path.endswith((".py", ".pyc", ".so")) and path.startswith(_LIB_ROOTS) and not any(c in mode for c in "wax+"):
            return                                   # lazy imports of stdlib / site-packages modules (read-only)
        raise PermissionError(f"sandbox: file access blocked ({args[0]!r})")
    if ev.startswith(BLOCK_PREFIX):
        raise PermissionError(f"sandbox: blocked operation {ev}")
_STAGE = [0]
import sysconfig as _sc
_LIB_ROOTS = tuple({sys.prefix, sys.base_prefix, sys.exec_prefix, _sc.get_paths()["stdlib"], _sc.get_paths()["purelib"], _sc.get_paths()["platlib"]})
ns = {"np": np, "numpy": np, "math": math, "__builtins__": safe, "__name__": "generated"}
if want_torch: ns.update(torch=torch, nn=torch.nn, F=torch.nn.functional)
code_obj = compile(src, "<generated>", "exec")
sys.addaudithook(_hook)
exec(code_obj, ns)           # audit 'exec' allowed only for this top-level call
_STAGE[0] = 1
if fn_name not in ns or not callable(ns[fn_name]):
    names = [k for k, v in ns.items() if callable(v) and not k.startswith("_") and k not in ("np", "numpy", "math", "torch", "nn", "F")]
    print("SANDBOX_MISSING_FN " + json.dumps(names), file=sys.stderr); sys.exit(3)
W = ns[fn_name]() if fn_name == "build_weights" else ns[fn_name](dict(INPUTS))
def _arr(v):
    if hasattr(v, "detach"): v = v.detach().cpu().numpy()
    return np.asarray(v, dtype=np.float64)
if not isinstance(W, dict):
    print("SANDBOX_NOT_DICT " + type(W).__name__, file=sys.stderr); sys.exit(4)
arrs = {str(k): _arr(v) for k, v in W.items()}
_ALLOW[0] = True
np.savez(out_path, **arrs)
'''

class SandboxError(RuntimeError):
    def __init__(self, kind, msg): super().__init__(f"[{kind}] {msg}"); self.kind = kind

FENCE_RE = re.compile(r"(?:^|\n)[ \t]*(```|~~~)[ \t]*([A-Za-z0-9_+-]*)[^\n]*\n(.*?)(?:\n[ \t]*\1[ \t]*(?=\n|$)|\Z)", re.S)

def _compiles(s):
    try: compile(s, "<x>", "exec"); return True
    except SyntaxError: return False

def extract_code(text, fn_name="build_weights", stop_reason=None):
    """Return (code, info). Handles ```python/```py/``` fences, multiple fences, leading/trailing prose, unclosed (truncated) fences.
    Never strips imports. info = {method, n_fences, truncated, compiles}."""
    text = (text or "").replace("\r\n", "\n")
    info = {"n_fences": 0, "truncated": stop_reason == "max_tokens", "method": None, "compiles": False}
    blocks = []
    for m in FENCE_RE.finditer(text):
        lang, body = m.group(2).lower(), m.group(3)
        closed = bool(re.search(r"\n[ \t]*" + re.escape(m.group(1)) + r"[ \t]*$", m.group(0)))
        if lang in ("", "python", "py", "python3", "numpy"): blocks.append((body, closed))
    info["n_fences"] = len(blocks)
    if any(not c for _, c in blocks): info["truncated"] = True
    cands = []
    if blocks:
        with_fn = [b for b, _ in blocks if re.search(rf"def\s+{fn_name}\s*\(", b)]
        # several blocks: concatenate (helpers may live in other blocks; a later re-definition wins, i.e. the final version)
        if len(blocks) > 1: cands.append(("fences_concat", "\n\n".join(b for b, _ in blocks)))
        if with_fn: cands.append(("fence_with_fn", with_fn[-1]))
        cands.append(("longest_fence", max((b for b, _ in blocks), key=len)))
    else:
        lines = text.split("\n")
        start = next((i for i, l in enumerate(lines) if re.match(r"\s*(import |from \w|def |@|[A-Za-z_]\w*\s*=)", l)), None)
        if start is not None:
            body = lines[start:]
            for end in range(len(body), 0, -1):          # trim trailing prose until it compiles
                s = "\n".join(body[:end])
                if _compiles(s): cands.append(("unfenced", s)); break
            else: cands.append(("unfenced_raw", "\n".join(body)))
        else: cands.append(("raw", text))
    for method, s in cands:
        if _compiles(s) and re.search(rf"def\s+{fn_name}\s*\(", s):
            info.update(method=method, compiles=True); return s, info
    method, s = cands[0]
    info.update(method=method, compiles=_compiles(s)); return s, info

def classify_error(msg):
    m = str(msg)
    for kind, pats in [("truncated", ["[truncated]"]), ("timeout", ["[timeout]", "TimeoutExpired"]),
                       ("blocked", ["sandbox: ", "is blocked in the sandbox"]), ("syntax", ["SyntaxError", "IndentationError"]),
                       ("missing_fn", ["SANDBOX_MISSING_FN", "[missing_fn]"]), ("not_dict", ["SANDBOX_NOT_DICT", "weights must be a dict"]),
                       ("shape", ["shape", "missing parameter"]), ("nonfinite", ["non-finite"]), ("name_error", ["NameError"]),
                       ("import", ["ImportError", "ModuleNotFoundError"]), ("no_code", ["[no_code]"])]:
        if any(p in m for p in pats): return kind
    return "runtime"

def run_generated(code_str, fn_name="build_weights", inputs=None, timeout=120, want_torch=None):
    """Run generated code in a SUBPROCESS. Returns dict of float64 arrays. Raises SandboxError(kind, ...)."""
    if want_torch is None: want_torch = bool(re.search(r"\btorch\b", code_str))
    if not _compiles(code_str):
        try: compile(code_str, "<generated>", "exec")
        except SyntaxError as e: raise SandboxError("syntax", f"SyntaxError: {e}")
    d = tempfile.mkdtemp(); src, out, inp, run = f"{d}/g.py", f"{d}/w.npz", f"{d}/in.npz", f"{d}/runner.py"
    open(src, "w").write(code_str); open(run, "w").write(RUNNER)
    np.savez(inp, **(inputs or {"_": np.zeros(1)}))
    env = {k: v for k, v in os.environ.items() if k not in ("ANTHROPIC_API_KEY", "HF_TOKEN", "KAGGLE_KEY")}
    env.update(OMP_NUM_THREADS="4", CUDA_VISIBLE_DEVICES="")
    try:
        r = subprocess.run([sys.executable, "-I", run, src, out, fn_name, inp, "1" if want_torch else "0"],
                           capture_output=True, text=True, timeout=timeout, cwd=d, env=env)
    except subprocess.TimeoutExpired:
        raise SandboxError("timeout", f"generated code exceeded {timeout}s")
    if r.returncode != 0:
        err = r.stderr[-800:]
        raise SandboxError(classify_error(err), "generated code failed: " + err)
    return {k: v for k, v in np.load(out).items()}

def validate(W, shapes):
    if not isinstance(W, dict): raise SandboxError("not_dict", "weights must be a dict")
    out = {}
    for k, shp in shapes.items():
        if k not in W: raise SandboxError("shape", f"missing parameter {k} (got {sorted(W)[:10]})")
        a = np.asarray(W[k], dtype=np.float64)
        if a.shape != tuple(shp) and a.size == int(np.prod(shp)): a = a.reshape(shp)
        if a.shape != tuple(shp): raise SandboxError("shape", f"{k} shape {a.shape} != {tuple(shp)}")
        if not np.isfinite(a).all(): raise SandboxError("nonfinite", f"{k} has non-finite values")
        out[k] = a
    return out

def load_weights(model, W):
    sd = model.state_dict()
    with torch.no_grad():
        for k in sd: sd[k].copy_(torch.tensor(W[k], dtype=sd[k].dtype))
    return model.to(DEVICE)

def get_weights(model): return {k: v.detach().cpu().numpy().astype(np.float64) for k, v in model.state_dict().items()}

def margin_stats(lg, y):
    """Raw logit margin (true - best other), margin normalised by per-example logit std (scale-free), and softmax-probability margin."""
    lg = lg.double(); c = lg.gather(1, y[:, None]).squeeze(1); o = lg.clone(); o.scatter_(1, y[:, None], -float("inf"))
    raw = c - o.max(-1).values
    sd = lg.std(-1).clamp_min(1e-12); norm = raw / sd
    p = torch.softmax(lg, -1); pc = p.gather(1, y[:, None]).squeeze(1); po = p.clone(); po.scatter_(1, y[:, None], -1.0)
    return raw, norm, pc - po.max(-1).values

@torch.no_grad()
def evaluate(model, X, Y, idx=None, noise=0.0, seed=0, bs=16384):
    model.eval().to(DEVICE)
    if idx is not None: X, Y = X[idx], Y[idx]
    g = torch.Generator().manual_seed(seed); preds, raws, norms, pms, losses = [], [], [], [], []
    for i in range(0, len(Y), bs):
        x = X[i:i+bs]
        if noise > 0: x = x + noise * torch.randn(x.shape, generator=g)
        x, y = x.to(DEVICE), Y[i:i+bs].to(DEVICE)
        lg = torch.nan_to_num(model(x).float(), nan=0.0, posinf=1e30, neginf=-1e30)
        preds.append(lg.argmax(-1).cpu()); r, n, pm = margin_stats(lg, y)
        raws.append(r.cpu()); norms.append(n.cpu()); pms.append(pm.cpu())
        losses.append(F.cross_entropy(lg.double(), y, reduction="sum").item())
    pred, r, n, pm = torch.cat(preds), torch.cat(raws), torch.cat(norms), torch.cat(pms)
    wrong = (pred != Y.cpu()).nonzero().squeeze(1)
    return {"acc": (pred == Y.cpu()).float().mean().item(), "loss": sum(losses) / len(Y),
            "mean_margin_raw": r.mean().item(), "min_margin_raw": r.min().item(),
            "mean_margin_norm": n.mean().item(), "min_margin_norm": n.min().item(),
            "mean_prob_margin": pm.mean().item(), "min_prob_margin": pm.min().item(),
            "wrong": wrong[:2000].tolist(), "pred": pred}

def fourier_energy(task, W):
    if task not in ("modadd", "modmul", "permadd") or "W1" not in W: return float("nan")
    E = W["W1"][:, :P]
    if task == "modmul": E = E[:, [pow(G_ROOT, e, P) for e in range(P - 1)]]
    if task == "permadd": E = E[:, np.argsort(PERM)]      # columns ordered by underlying value
    pw = (np.abs(np.fft.rfft(E, axis=1)) ** 2).sum(0)[1:]
    return float(np.sort(pw)[::-1][:10].sum() / (pw.sum() + 1e-12))

def structure(task, W):
    out = {}
    for name, A in W.items():
        if A.ndim < 2: continue
        A2 = A.reshape(A.shape[0], -1); s = np.linalg.svd(A2, compute_uv=False)
        if s[0] <= 0: out.update({f"{name}_rank": 0, f"{name}_eff_rank": 0.0, f"{name}_sparsity": 1.0}); continue
        q = s / s.sum()
        out[f"{name}_rank"] = int((s > 1e-3 * s[0]).sum()); out[f"{name}_eff_rank"] = float(np.exp(-(q @ np.log(q + 1e-12))))
        out[f"{name}_sparsity"] = float((np.abs(A2) < 1e-3 * np.abs(A2).max()).mean()); out[f"{name}_fro"] = float(np.linalg.norm(A2))
    out["fourier_top10_energy"] = fourier_energy(task, W)
    return out

NOISES = [0.1, 0.3, 0.5]
MARGIN_KEYS = ["mean_margin_raw", "min_margin_raw", "mean_margin_norm", "min_margin_norm", "mean_prob_margin", "min_prob_margin"]
def full_metrics(task, model, T, sp=None):
    r_all = evaluate(model, T["X"], T["Y"])
    row = {"all_acc": r_all["acc"], **{k: r_all[k] for k in MARGIN_KEYS}}
    if sp is not None: row["test_acc"] = evaluate(model, T["X"], T["Y"], sp[2])["acc"]
    for nz in NOISES: row[f"acc_noise{nz}"] = evaluate(model, T["X"], T["Y"], noise=nz)["acc"]
    row.update(structure(task, get_weights(model))); return row

def t_ci(x):
    x = np.asarray([v for v in x if v is not None and np.isfinite(v)], float); n = len(x)
    if n == 0: return np.nan, np.nan, np.nan
    if n == 1: return x[0], 0.0, 0.0
    try:
        from scipy import stats; tq = stats.t.ppf(0.975, n - 1)
    except Exception: tq = 1.96
    return x.mean(), x.std(ddof=1), tq * x.std(ddof=1) / math.sqrt(n)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; d = 1 + z * z / n; c = (ph + z * z / (2 * n)) / d; h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

PER_RUN, A_CURVES, C_ROWS = [], [], []
def save_all():
    pd.DataFrame(PER_RUN).drop(columns=["pred"], errors="ignore").to_csv(f"{OUT_DIR}/per_run.csv", index=False)
    pd.DataFrame(A_CURVES).to_csv(f"{OUT_DIR}/A_curves.csv", index=False)
    pd.DataFrame(C_ROWS).to_csv(f"{OUT_DIR}/C_edits.csv", index=False)
    json.dump({"per_run": PER_RUN, "C": C_ROWS, "llm_usage": globals().get("USAGE", {}), "elapsed_s": time.time() - T_START},
              open(f"{OUT_DIR}/results_partial.json", "w"), default=str)

ORACLE_W = {}
if RUN_ORACLE_CHECK:
    for t, T in TASKS.items():
        W = validate(run_generated(ORACLE_SRC[t]), T["shapes"]); ORACLE_W[t] = W
        m = load_weights(T["make_model"](), W); r = full_metrics(t, m, T)
        assert r["all_acc"] == 1.0, f"oracle {t} not exact: {r['all_acc']}"
        PER_RUN.append({"task": t, "instance": "B_oracle", "synth": "oracle", "synth_model": "analytic", "prompt": "-", "rep": 0, "ok": True, "success": True, **r})
        print(f"oracle {t}: acc {r['all_acc']:.4f} | norm min margin {r['min_margin_norm']:.3f} | prob margin {r['min_prob_margin']:.4f}")
    save_all()


In [ ]:
# ---------------- Instance B: one synthesis attempt -> extraction -> sandbox -> validation -> metrics ----------------
def label_of(syn):
    """Row label = model that actually produced the reply (never the requested-but-unavailable one)."""
    return syn.get("resolved_id") or syn["model_id"]

def process_reply(task, text, meta, fn_base, shapes=None, cifar=False):
    """Shared by live runs and offline replay. Returns row fields (ok, all_acc, error, error_kind, extraction info, metrics)."""
    row = {"ok": False, "all_acc": 0.0, "error": None, "error_kind": None}
    code_str, info = extract_code(text, "build_weights", meta.get("stop_reason"))
    row.update({f"extract_{k}": v for k, v in info.items()})
    open(f"{GEN_DIR}/{fn_base}.py", "w").write(code_str)
    try:
        if not re.search(r"def\s+build_weights\s*\(", code_str):
            raise SandboxError("truncated" if info["truncated"] else "no_code", "no build_weights definition in reply")
        W = validate(run_generated(code_str, timeout=SANDBOX_TIMEOUT["cifar" if cifar else "algo"]), shapes or TASKS[task]["shapes"])
        if cifar:
            m = load_weights(SmallCNN(), W); row.update(cifar_metrics(m, CIFAR_DATA[2], CIFAR_DATA[3])); row["ok"] = True
            row["_model"] = m
        else:
            T = TASKS[task]; m = load_weights(T["make_model"](), W); row.update(full_metrics(task, m, T)); row["ok"] = True
        np.savez_compressed(f"{GEN_DIR}/{fn_base}_weights.npz", **W)
    except SandboxError as e:
        row["error"], row["error_kind"] = str(e)[:500], e.kind
    except Exception as e:
        row["error"], row["error_kind"] = repr(e)[:500], classify_error(repr(e))
    if info["truncated"] and not row["ok"]: row["error_kind"] = "truncated"
    if cifar: row["success"] = None           # accuracy only for CIFAR
    else: row["success"] = bool(row["ok"] and row["all_acc"] >= SUCCESS_THRESH)
    return row



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import model_info
from datetime import datetime
from zoneinfo import ZoneInfo

def digest(text):return hashlib.sha256(text.encode()).hexdigest()
def append_json(path, obj):
    with open(path,'a') as f:f.write(json.dumps(obj,default=str)+'\n');f.flush()
def event(**kw):
    append_json(OUT_DIR+'/events.jsonl',{'time_dhaka':datetime.now(ZoneInfo('Asia/Dhaka')).isoformat(),**kw})
def check_runtime():
    if torch.cuda.device_count()!=2:raise RuntimeError('Requires two CUDA GPUs; no CPU synthesis fallback')
    for i in range(2):
        if torch.cuda.get_device_properties(i).total_memory<14*2**30:raise RuntimeError('GPU memory below two T4-class cards')
    print(subprocess.check_output(['nvidia-smi'],text=True),flush=True)

def run_model(spec):
    cache=Path(tempfile.gettempdir())/('wsp_disposable_hf_'+digest(spec['id'])[:10])
    # This function owns only this new private model-cache directory; no shared HF cache touched.
    event(event='DISK_PREFLIGHT',model=spec['id'],cache_parent=str(cache.parent),free_bytes=shutil.disk_usage(cache.parent).free,required_gib=spec['disk_gib'])
    if cache.exists():raise RuntimeError('Private model cache already exists; inspect previous run before cleanup')
    if shutil.disk_usage(cache.parent).free<spec['disk_gib']*2**30:
        event(event='SKIPPED',model=spec['id'],reason='insufficient_download_disk',required_gib=spec['disk_gib']);return
    cache.mkdir()
    model=tok=None
    try:
        info=model_info(spec['id'],revision=spec['revision'],token=False)
        if info.gated:raise RuntimeError('Model unexpectedly gated: no authentication attempted')
        tok=AutoTokenizer.from_pretrained(spec['id'],revision=spec['revision'],cache_dir=str(cache),token=False,trust_remote_code=False)
        # Chat-template preflight before downloading weights.
        if not tok.chat_template:raise RuntimeError('Missing official chat template')
        probe=tok.apply_chat_template([{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':task_prompt('modmul','none')}],tokenize=True,add_generation_prompt=True)
        if isinstance(probe,dict) or hasattr(probe,'keys'):probe=probe['input_ids']
        if probe and isinstance(probe[0],list):probe=probe[0]
        if len(probe)<100:raise RuntimeError('Implausibly short tokenized prompt')
        event(event='TOKENIZER_READY',model=spec['id'],revision=info.sha,prompt_tokens=len(probe),chat_template=tok.chat_template,chat_template_sha256=digest(tok.chat_template))
        q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
        model=AutoModelForCausalLM.from_pretrained(spec['id'],revision=spec['revision'],cache_dir=str(cache),token=False,trust_remote_code=False,quantization_config=q,device_map='balanced',max_memory={0:'12GiB',1:'12GiB'},torch_dtype=torch.float16,attn_implementation='sdpa',low_cpu_mem_usage=True)
        if any(str(v) in ('cpu','disk') for v in model.hf_device_map.values()):raise RuntimeError('CPU/disk offload rejected; insufficient GPU fit')
        model.eval()
        model.generation_config.save_pretrained(OUT_DIR+'/'+digest(spec['id'])[:10]+'_generation_config')
        event(event='MODEL_READY',model=spec['id'],revision=info.sha,device_map=model.hf_device_map,footprint_bytes=model.get_memory_footprint(),quantization=q.to_dict(),model_config=model.config.to_dict())
        for task,level,rep in PILOT_CASES:
            if time.time()-T_START>MAX_TOTAL_S:
                event(event='TIME_BUDGET_STOP',model=spec['id']);break
            tag=re.sub(r'[^A-Za-z0-9_.-]','_',spec['id'])+f'_{task}_{level}_r{rep}'
            raw_path=Path(OUT_DIR)/(tag+'.json')
            if raw_path.exists():raise RuntimeError('Existing response; refusing to overwrite '+str(raw_path))
            user=task_prompt(task,level)
            msgs=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user}]
            rendered=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
            ids=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors='pt')
            if not torch.is_tensor(ids):ids=ids['input_ids']
            if ids.shape[1]+MAX_NEW_TOKENS>model.config.max_position_embeddings:raise RuntimeError('Context limit exceeded; no prompt truncation allowed')
            ids=ids.to(model.get_input_embeddings().weight.device);seed=1000+rep;set_seed(seed)
            start=time.perf_counter()
            for i in range(2):torch.cuda.reset_peak_memory_stats(i)
            try:
                with torch.inference_mode():
                    out=model.generate(ids,attention_mask=torch.ones_like(ids),max_new_tokens=MAX_NEW_TOKENS,max_time=MAX_GENERATION_S,pad_token_id=tok.eos_token_id,**SAMPLING)
                gen=out[0,ids.shape[1]:]; token_ids=gen.tolist(); text=tok.decode(gen,skip_special_tokens=True)
                eos=model.generation_config.eos_token_id or tok.eos_token_id
                eos=eos if isinstance(eos,list) else [eos]
                stop='end_turn' if token_ids and token_ids[-1] in eos else ('max_tokens' if len(token_ids)>=MAX_NEW_TOKENS else 'time_limit')
                meta={'in_tok':int(ids.shape[1]),'out_tok':len(token_ids),'stop_reason':stop,'reply_source':'local','cost_usd':0.0,'latency_s':time.perf_counter()-start}
                record={'model_id':spec['id'],'revision':info.sha,'task':task,'prompt_level':level,'rep':rep,'seed':seed,'sampling':SAMPLING,'max_new_tokens':MAX_NEW_TOKENS,'max_generation_s':MAX_GENERATION_S,'messages':msgs,'rendered_prompt':rendered,'prompt_sha256':digest(rendered),'chat_template_sha256':digest(tok.chat_template),'response':text,'response_sha256':digest(text),'generated_token_ids':token_ids,'meta':meta,'peak_allocated_bytes':[torch.cuda.max_memory_allocated(i) for i in range(2)]}
                raw_path.write_text(json.dumps(record,indent=2)) # preserve raw BEFORE evaluation
                row={'task':task,'instance':'B','synth':spec['id'],'revision':info.sha,'prompt':level,'rep':rep,**meta,**process_reply(task,text,meta,tag)}
                row['incomplete']=stop!='end_turn'
                PER_RUN.append(row);save_all()
                print(tag,'valid',row['ok'],'accuracy',row['all_acc'],'stop',stop,flush=True)
                del out,gen
                if stop=='time_limit':
                    event(event='SLOW_GENERATION_STOP',model=spec['id'],tag=tag);break
            except Exception as e:
                event(event='GENERATION_ERROR',model=spec['id'],task=task,rep=rep,error=repr(e),traceback=traceback.format_exc());break
            finally:
                del ids;gc.collect();torch.cuda.empty_cache()
    except Exception as e:
        event(event='MODEL_ERROR',model=spec['id'],error=repr(e),traceback=traceback.format_exc())
        print('MODEL_ERROR',spec['id'],repr(e),flush=True)
    finally:
        del model,tok;gc.collect();torch.cuda.empty_cache()
        shutil.rmtree(cache) # only the exact disposable directory created above

check_runtime()
manifest={'python':sys.version,'platform':platform.platform(),'packages':{p:importlib.metadata.version(p) for p in ['torch','transformers','accelerate','bitsandbytes','mistral-common','numpy']},'models':MODELS,'planned_cases':PILOT_CASES,'max_new_tokens':MAX_NEW_TOKENS,'max_generation_s':MAX_GENERATION_S,'sampling':SAMPLING,'gpus':[torch.cuda.get_device_properties(i).name for i in range(2)],'note':'pilot only; no provider API usage; fixed permadd mapping; FP32 CPU evaluator'}
Path(OUT_DIR+'/manifest.json').write_text(json.dumps(manifest,indent=2))
for spec in MODELS:
    if time.time()-T_START>MAX_TOTAL_S:break
    run_model(spec)
# Keep incomplete and infrastructure failures distinguishable; do not silently pool with completed outputs.
B=pd.DataFrame([r for r in PER_RUN if r.get('instance')=='B'])
if len(B):
    summary=B.groupby(['synth','task','prompt'],dropna=False).agg(n=('success','size'),valid=('ok','sum'),successes=('success','sum'),mean_acc=('all_acc','mean'),incomplete=('incomplete','sum'))
    summary.to_csv(OUT_DIR+'/pilot_summary.csv');print(summary)
print('Pilot finished. Inspect events.jsonl for missing/aborted attempts. Not a completed benchmark.',flush=True)
